# Diário de Bordo automatizado

Gera o **Diário de Bordo em PDF** a partir da planilha de ponto exportada do sistema
(colunas `Data, Hora, Nome, Ação, Timestamp`).

1. Execute as células na ordem (**Ambiente de execução → Executar tudo**).
2. Edite a célula de configuração com o nome da empresa, do setor e do funcionário.
3. Na última célula, envie o arquivo `.xlsx` ou `.csv`. O PDF é baixado automaticamente.

Há uma planilha de exemplo (dados fictícios) em `exemplos/ponto_exemplo.csv`.

In [ ]:
# 1. Instalação das bibliotecas
!pip install reportlab pandas ipywidgets openpyxl -q

In [ ]:
# 2. Configuração: edite apenas estas linhas
EMPRESA = "Sua Empresa"
SETOR = "Setor de T.I"
NOME_FUNCIONARIO = "Nome do Funcionário"
ARQUIVO_PDF = "diario_de_bordo.pdf"

# Rodapé do diário (ajuste conforme a política da sua empresa)
REGRAS_RODAPE = (
    "1- Registre todos os horários com exatidão.<br/>"
    "2- Respeite os intervalos de descanso e de refeição previstos na sua jornada.<br/>"
    "3- Respeite o repouso entre o fim de uma jornada e o início da seguinte."
)

In [ ]:
# 3. Leitura da planilha de ponto
import io
import pandas as pd

# Nomes aceitos para cada coluna (a planilha gerada pelo Apps Script usa: Data, Hora, Nome, Ação, Timestamp)
ALIASES = {
    "data": ["data"],
    "hora": ["hora", "horário", "horario"],
    "nome": ["nome"],
    "evento": ["ação", "acao", "evento"],
    "timestamp": ["timestamp", "data/hora", "datahora"],
}

def _normaliza(txt):
    return str(txt).strip().lower()

def _mapeia_colunas(df):
    """Mapeia as colunas por nome; se não houver cabeçalho, usa a ordem Data, Hora, Nome, Ação, Timestamp."""
    achadas = {}
    for col in df.columns:
        for chave, nomes in ALIASES.items():
            if _normaliza(col) in nomes and chave not in achadas:
                achadas[chave] = col
    if {"hora", "evento"} <= set(achadas):
        return achadas
    # sem cabeçalho reconhecido: posição (5 colunas) ou (4 colunas: Hora, Nome, Evento, Timestamp)
    cols = list(df.columns)
    if len(cols) >= 5:
        return dict(zip(["data", "hora", "nome", "evento", "timestamp"], cols[:5]))
    if len(cols) == 4:
        return dict(zip(["hora", "nome", "evento", "timestamp"], cols[:4]))
    raise ValueError("Formato de planilha não reconhecido.")

def processar_planilha_ponto(conteudo_bytes, e_csv=False):
    """Lê a planilha de ponto e devolve uma linha por dia, no formato do diário de bordo."""
    leitor = pd.read_csv if e_csv else pd.read_excel
    df = leitor(io.BytesIO(conteudo_bytes))
    if not any(_normaliza(c) in ALIASES["hora"] + ALIASES["evento"] for c in df.columns):
        df = leitor(io.BytesIO(conteudo_bytes), header=None)   # planilha sem cabeçalho
    col = _mapeia_colunas(df)

    # carimbo de data/hora: usa a coluna Timestamp; senão, junta Data + Hora
    if "timestamp" in col:
        df["_ts"] = pd.to_datetime(df[col["timestamp"]], errors="coerce")
    else:
        df["_ts"] = pd.to_datetime(df[col["data"]].astype(str) + " " + df[col["hora"]].astype(str), errors="coerce", dayfirst=True)
    df = df.dropna(subset=["_ts"]).sort_values("_ts")

    dias = {}
    for dia, grupo in df.groupby(df["_ts"].dt.date):
        data_str = dia.strftime("%d/%m/%y")
        rec = {k: "" for k in ["DATA", "INICIO_JORNADA",
               "MOTIVO_1", "PARADA1_INC", "PARADA1_FIM", "MOTIVO_2", "PARADA2_INC", "PARADA2_FIM",
               "MOTIVO_3", "PARADA3_INC", "PARADA3_FIM", "MOTIVO_4", "PARADA4_INC", "PARADA4_FIM"]}
        rec["DATA"] = data_str
        for _, linha in grupo.iterrows():
            hora = linha["_ts"].strftime("%H:%M:%S")
            evento = _normaliza(linha[col["evento"]])
            if "entrada" in evento:
                rec["INICIO_JORNADA"] = rec["INICIO_JORNADA"] or hora       # primeira entrada do dia
            elif "almoço" in evento or "almoco" in evento:
                if "retorno" in evento:
                    rec["PARADA1_FIM"] = hora
                    rec["MOTIVO_1"] = "Almoço"
                else:
                    rec["PARADA1_INC"] = hora
            elif "saída" in evento or "saida" in evento:
                rec["PARADA2_FIM"] = hora                                      # última saída do dia
        dias[data_str] = rec

    datas = list(dias)
    periodo = f"De {datas[0]} A {datas[-1]}" if datas else ""
    return list(dias.values()), periodo

In [ ]:
# 4. Geração do PDF (A4, retrato)
from reportlab.lib import colors
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import ParagraphStyle, getSampleStyleSheet
from reportlab.platypus import Paragraph, SimpleDocTemplate, Spacer, Table, TableStyle

def gerar_diario(caminho_pdf, registros, periodo):
    doc = SimpleDocTemplate(caminho_pdf, pagesize=A4, rightMargin=8, leftMargin=8, topMargin=10, bottomMargin=10)
    st = getSampleStyleSheet()
    s_emp = ParagraphStyle("Emp", parent=st["Normal"], fontSize=9, leading=11, fontName="Helvetica-Bold")
    s_tit = ParagraphStyle("Tit", parent=st["Normal"], fontSize=10, leading=12, alignment=1, fontName="Helvetica-Bold")
    s_sub = ParagraphStyle("Sub", parent=st["Normal"], fontSize=7, leading=9, fontName="Helvetica-Bold")
    s_h = ParagraphStyle("H", parent=st["Normal"], fontSize=5, leading=6, alignment=1, fontName="Helvetica-Bold")
    s_b = ParagraphStyle("B", parent=st["Normal"], fontSize=5.5, leading=7, alignment=1)
    s_r = ParagraphStyle("R", parent=st["Normal"], fontSize=5.5, leading=7)

    cab = Table([
        [Paragraph(f"{EMPRESA.upper()}<br/><font size=7>DIÁRIO DO {SETOR.upper()}</font>", s_emp),
         Paragraph("<b>DIÁRIO DE BORDO</b>", s_tit),
         Paragraph("<b>JORNADA NORMAL</b> &nbsp;&nbsp; <b>JORNADA EXTRA</b>", s_sub)],
        [Paragraph(f"<b>NOME:</b> {NOME_FUNCIONARIO.upper()}", st["Normal"]), Paragraph("", st["Normal"]),
         Paragraph(f"<b>Período:</b> {periodo}", st["Normal"])],
    ], colWidths=[200, 180, 200])
    cab.setStyle(TableStyle([("VALIGN", (0, 0), (-1, -1), "MIDDLE"), ("ALIGN", (1, 0), (1, 0), "CENTER"), ("ALIGN", (2, 0), (2, -1), "RIGHT")]))

    topo = [["DATA", "INÍCIO\nJORNADA", "1ª Parada", "", "", "2ª Parada", "", "", "3ª Parada", "", "", "4ª Parada", "", ""],
            ["", "", "Motivo", "Hor. Início", "Hor. Fim", "Motivo", "Hor. Início", "Hor. Fim", "Motivo", "Hor. Início", "Hor. Fim", "Motivo", "Hor. Início", "Hor. Fim"]]
    conteudo = [[Paragraph(c.replace("\n", "<br/>"), s_h) for c in linha] for linha in topo]
    chaves = ["DATA", "INICIO_JORNADA", "MOTIVO_1", "PARADA1_INC", "PARADA1_FIM", "MOTIVO_2", "PARADA2_INC", "PARADA2_FIM",
              "MOTIVO_3", "PARADA3_INC", "PARADA3_FIM", "MOTIVO_4", "PARADA4_INC", "PARADA4_FIM"]
    for item in registros:
        conteudo.append([Paragraph(str(item.get(k, "")), s_b) for k in chaves])

    tab = Table(conteudo, colWidths=[42, 42, 32, 45, 45, 32, 45, 45, 32, 45, 45, 32, 45, 45], repeatRows=2)
    tab.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, 1), colors.HexColor("#F0F0F0")),
        ("ALIGN", (0, 0), (-1, -1), "CENTER"), ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),
        ("GRID", (0, 0), (-1, -1), 0.5, colors.black),
        ("SPAN", (0, 0), (0, 1)), ("SPAN", (1, 0), (1, 1)),
        ("SPAN", (2, 0), (4, 0)), ("SPAN", (5, 0), (7, 0)), ("SPAN", (8, 0), (10, 0)), ("SPAN", (11, 0), (13, 0)),
    ]))
    ass = Table([["_" * 41, "_" * 41], ["Gestor do Departamento", "Funcionário"]], colWidths=[290, 290])
    ass.setStyle(TableStyle([("ALIGN", (0, 0), (-1, -1), "CENTER"), ("FONTSIZE", (0, 1), (-1, 1), 7)]))
    doc.build([cab, Spacer(1, 6), tab, Spacer(1, 6), Paragraph(REGRAS_RODAPE, s_r), Spacer(1, 10), ass])

In [ ]:
# 5. Interface: envie a planilha (.xlsx ou .csv) e o PDF é baixado automaticamente
import ipywidgets as widgets
from IPython.display import display
from google.colab import files

upload = widgets.FileUpload(accept=".xlsx,.csv", multiple=False)
saida = widgets.Output()

def ao_enviar(change):
    with saida:
        saida.clear_output()
        if not upload.value:
            return
        item = upload.value[0] if isinstance(upload.value, (list, tuple)) else list(upload.value.values())[0]
        nome = item["name"]
        conteudo = bytes(item["content"])
        print(f"Processando {nome}...")
        try:
            registros, periodo = processar_planilha_ponto(conteudo, e_csv=nome.lower().endswith(".csv"))
            if not registros:
                print("Nenhum registro válido encontrado na planilha.")
                return
            gerar_diario(ARQUIVO_PDF, registros, periodo)
            print(f"PDF gerado: {len(registros)} dia(s), {periodo}.")
            files.download(ARQUIVO_PDF)
        except Exception as e:
            print(f"Erro ao processar o arquivo: {e}")

upload.observe(ao_enviar, names="value")
print("=== AUTOMATIZADOR DE DIÁRIO DE BORDO ===")
print("Envie a planilha de ponto (.xlsx ou .csv):")
display(upload, saida)